# IMPORTS

In [ ]:
from pathlib import Path
import pandas as pd
from diive.core.io.files import save_parquet, load_parquet

In [ ]:
# Project settings (dataset period, data paths): see src/config.py
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.config import DATA_DIR, PERIOD_START, PERIOD_END, select_period
print(f"Dataset period: {PERIOD_START} to {PERIOD_END} (CET, UTC+1)")

# LOAD DATA

In [ ]:
maindf = load_parquet(filepath=r"../70_SPLIT_TREATMENTS/71.1_FLUXES_L3.3_NEE_LE_H_FN2O_FCH4_REDDYPROC_PARCELS.parquet")
gf_part = load_parquet(filepath=r"../80_GAP-FILLING/84.1_GF-FLUXES+PARTITIONED.parquet")
meteo = load_parquet(filepath=r"../10_METEO/13.1_CH-TAN_meteo_gapfilled.parquet")
management = load_parquet(filepath=r"../20_MANAGEMENT/21.1_mgmt_full_timestamp.parquet")

# MERGE

## Gap-filled & partitioned fluxes

In [ ]:
newcols = [c for c in gf_part.columns if c not in maindf]
print("NEW VARIABLES FROM FLUX PROCESSING CHAIN:")
[print(f"+ {c}") for c in newcols]
maindf = pd.concat([maindf, gf_part[newcols]], axis=1)
maindf

## Meteo

In [ ]:
newcols = [c for c in meteo.columns if c not in maindf]
print("NEW VARIABLES FROM METEO DATA:")
[print(f"+ {c}") for c in newcols]
maindf = pd.concat([maindf, meteo[newcols]], axis=1)
maindf

## Management

In [ ]:
newcols = [c for c in management.columns if c not in maindf]

# drop any that start with 'n_decay' or 'timesince' OR end with 'timed'
newcols = [
    c for c in newcols
    if not (c.startswith(('n_decay', 'timesince')) or c.endswith('timed'))
]

print("NEW VARIABLES FROM MANAGEMENT DATA:")
[print(f"+ {c}") for c in newcols]
maindf = pd.concat([maindf, management[newcols]], axis=1)
maindf

# CHECK DATASET PERIOD

In [ ]:
# Restrict the product to the dataset period (safety cut).
# All inputs should already be restricted from the first notebooks on; if records outside
# the period are found here, an upstream output is outdated and should be re-run.
n_before = len(maindf)
outside = maindf.index[(maindf.index <= PERIOD_START) | (maindf.index >= PERIOD_END)]
if len(outside):
    print(f"WARNING: {len(outside)} records outside the dataset period were removed "
          f"({outside.min()} to {outside.max()}). Check that all upstream notebooks were re-run.")
maindf = select_period(maindf, timestamp='middle')

# Checks
assert maindf.index.is_unique, "Duplicated timestamps in the product"
expected = pd.date_range(PERIOD_START + pd.Timedelta('15min'), PERIOD_END - pd.Timedelta('15min'), freq='30min')
missing = expected.difference(maindf.index)
print(f"{len(maindf)} records from {maindf.index.min()} to {maindf.index.max()}; "
      f"missing 30-min timestamps: {len(missing)}")

# CORE DATASET

The full product contains all variables of all processing steps (> 2000 columns). The core dataset contains the variables most users need:

- **Fluxes** NEE, FN2O, FCH4: u\*-filtered, quality-controlled measured fluxes and XGBoost gap-filled fluxes for the whole footprint and for parcels A and B, with gap-filling flags and uncertainties
- **NEE partitioning**: GPP and RECO (nighttime method) from the gap-filled NEE
- **LE, H**: measured (quality-controlled) and MDS gap-filled
- EddyPro random uncertainties, u\*, wind, u\* threshold, parcel attribution
- **Meteo and soil**: gap-filled meteorological variables (with MeteoSwiss flags) and gap-filled soil temperature and water content
- **Management**

Settings below define which u\* scenario and quality level are included. Columns that do not exist in the data are reported and skipped.

In [ ]:
# --- Settings ---------------------------------------------------------------
USTAR_CUTS = ['16', '50', '84']        # u* scenarios to include, e.g. ['16', '50', '84']
QC_LEVELS = ['QCF0', 'QCF']       # 'QCF0' = highest quality only, 'QCF' = including flag 1
GASES = ['NEE', 'FN2O', 'FCH4']

# --- Column selection -----------------------------------------------------
core_cols = []

# Parcel attribution, turbulence and wind
core_cols += ['parcel', 'parcel_certainty', 'SW_IN_POT', 'USTAR', 'WS', 'WD',
              'Ustar_U50_Thres_reddyproc']

# Gas fluxes: measured, gap-filled, flags and uncertainty
for gas in GASES:
    for cut in USTAR_CUTS:
        for qc in QC_LEVELS:
            for area in ['footprint', 'parcelA', 'parcelB']:
                base = f'{gas}_L3.3_CUT_{cut}_{qc}_{area}'
                core_cols += [
                    base,                                     # measured, u*-filtered
                    f'{base}_gfXGBoost',                      # gap-filled
                    f'FLAG_{base}_gfXGBoost_ISFILLED',        # 1 = gap-filled value
                    f'{base}_gfXGBoost_sigmaGF',              # gap-filling uncertainty
                    f'{base}_gfXGBoost_total_unc',            # total uncertainty
                ]

# NEE partitioning (from gap-filled NEE; partitioning is done for the CUT_50 scenario only)
for qc in QC_LEVELS:
    for var in ['GPP_NT', 'RECO_NT']:
        core_cols.append(f'{var}_CUT_50_{qc}_footprint_gfXGBoost')

# Energy fluxes: measured (quality-controlled) and MDS gap-filled
core_cols += ['LE_L3.1_L3.2_QCF', 'LE_L3.2_QCF_footprint_gfMDS',
              'H_L3.1_L3.2_QCF', 'H_L3.2_QCF_footprint_gfMDS']

# Meteo (gap-filled with MeteoSwiss, see *_is_meteoswiss flags; Source: MeteoSwiss)
meteo_vars = ['ta', 'rh', 'vpd', 'pa', 'prec', 'sw_in', 'ppfd']
core_cols += meteo_vars
core_cols += [f'{v}_is_meteoswiss' for v in meteo_vars]

# Soil temperature and water content (gap-filled with XGBoost)
for var in ['ts', 'swc']:
    for depth in ['0.05', '0.15', '0.3']:
        core_cols += [f'{var}_{depth}_gfXG', f'FLAG_{var}_{depth}_gfXG_ISFILLED']

# Management
core_cols += [c for c in management.columns
              if not (c.startswith(('n_decay', 'timesince')) or c.endswith('timed'))]

# --- Build core dataframe ---------------------------------------------------
core_cols = list(dict.fromkeys(core_cols))               # remove duplicates, keep order
missing_cols = [c for c in core_cols if c not in maindf.columns]
core_cols = [c for c in core_cols if c in maindf.columns]

core = maindf[core_cols].copy()
# Add explicit start/end timestamps of each 30-min record (index is TIMESTAMP_MIDDLE, CET)
core.insert(0, 'TIMESTAMP_START', core.index - pd.Timedelta('15min'))
core.insert(1, 'TIMESTAMP_END', core.index + pd.Timedelta('15min'))

print(f"Core dataset: {core.shape[1]} columns, {core.shape[0]} records")
if missing_cols:
    print(f"\nNot found in data, skipped ({len(missing_cols)}):")
    [print(f"- {c}") for c in missing_cols]
core

# SUBSET FOR N2O-PRECISION-FERTILIZATION

Input file of the analysis repository [n2o-precision-fertilization](https://github.com/fabioturc/n2o-precision-fertilization)
(there: `data/raw/fluxes_meteo_mgmt.parquet`). It contains exactly the variables its notebooks and R scripts read:

- **Turbulence, wind and parcel attribution** (footprint model input, day/night and parcel masks)
- **Measured fluxes** NEE, FN2O, FCH4 (`L3.3_CUT_50_QCF0`) for the whole footprint and per parcel
- **Gap-filled fluxes** (`QCF`, footprint / parcel A / parcel B) for the three u\* scenarios with gap-filling flags, and for `CUT_50`
  the uncertainty terms used for the budgets: `sigmaObs`, `sigmaResid`, `sigmaGF`, `sigmaEns`, `total_unc` and the
  cross-validation ensemble members `_ens##` (needed for the correlation-aware cumulative uncertainty)
- **Meteo and soil**: air temperature, precipitation, measured and gap-filled soil temperature and water content

In contrast to the core dataset, a missing variable raises an error here, so that the analysis repository
never receives an incomplete file. If a variable is added to the analysis, add it to the lists below.

In [ ]:
# --- Settings ---------------------------------------------------------------
N2O_GASES = ['NEE', 'FN2O', 'FCH4']
N2O_AREAS = ['footprint', 'parcelA', 'parcelB']
N2O_USTAR_CUTS = ['16', '50', '84']

# --- Column selection -----------------------------------------------------
n2o_cols = []

# Turbulence, wind and parcel attribution (footprint model input: scripts/00_prep_footprint_input.R)
n2o_cols += ['WS', 'USTAR', 'WD', 'MO_LENGTH', 'V_SIGMA', 'SW_IN_POT', 'parcel', 'parcel_certainty']

# Measured fluxes, u*-filtered, highest quality (CUT_50, QCF0)
for gas in N2O_GASES:
    n2o_cols.append(f'{gas}_L3.3_CUT_50_QCF0')                          # whole footprint, before parcel split
    n2o_cols += [f'{gas}_L3.3_CUT_50_QCF0_{area}' for area in N2O_AREAS]
n2o_cols.append('FN2O_L3.3_CUT_50_QCF_footprint')                    # footprint model: records with valid N2O flux

# Gap-filled fluxes (QCF) for all u* scenarios, with flags
for gas in N2O_GASES:
    for area in N2O_AREAS:
        for cut in N2O_USTAR_CUTS:
            base = f'{gas}_L3.3_CUT_{cut}_QCF_{area}_gfXGBoost'
            n2o_cols += [base, f'FLAG_{base}_ISFILLED']

# Uncertainty terms of the CUT_50 gap-filled fluxes (budgets and cumulative uncertainty bands)
n2o_ens_cols = []
for gas in N2O_GASES:
    for area in N2O_AREAS:
        base = f'{gas}_L3.3_CUT_50_QCF_{area}_gfXGBoost'
        n2o_cols += [f'{base}_{term}' for term in ['sigmaObs', 'sigmaResid', 'sigmaGF', 'sigmaEns', 'total_unc']]
        ens = sorted(c for c in maindf.columns if c.startswith(f'{base}_ens') and c[len(base) + 4:].isdigit())
        if not ens:
            raise KeyError(f"No ensemble members '{base}_ens##' found in the product")
        n2o_ens_cols += ens
n2o_cols += n2o_ens_cols

# Meteo and soil
n2o_cols += ['ta', 'ta_is_meteoswiss', 'prec', 'prec_is_meteoswiss']
for var in ['ts', 'swc']:
    for depth in ['0.05', '0.15', '0.3']:
        n2o_cols += [f'{var}_{depth}',                                  # measured
                     f'{var}_{depth}_gfXG', f'FLAG_{var}_{depth}_gfXG_ISFILLED']   # gap-filled

# --- Build subset (strict: all variables must exist) ----------------------
n2o_cols = list(dict.fromkeys(n2o_cols))
missing_n2o = [c for c in n2o_cols if c not in maindf.columns]
if missing_n2o:
    raise KeyError(f"Variables needed by n2o-precision-fertilization not found in the product: {missing_n2o}")

n2o_subset = maindf[n2o_cols].copy()
print(f"Subset for n2o-precision-fertilization: {n2o_subset.shape[1]} columns "
      f"({len(n2o_ens_cols)} ensemble members), {n2o_subset.shape[0]} records, "
      f"{n2o_subset.index.min()} to {n2o_subset.index.max()}")
n2o_subset

# EXPORT

In [ ]:
NAME = "CH-TAN_wheat_2023-24_FLUXES_METEO_MGMT"

# Full product: all variables, parquet and csv
filename_full = f"90.1_{NAME}_FULL"
save_parquet(data=maindf, filename=filename_full)
maindf.to_csv(f"{filename_full}.csv", index=True)
print(f"Saved {filename_full}.csv")

# Core product: parquet and csv
filename_core = f"90.2_{NAME}_CORE"
save_parquet(data=core, filename=filename_core)
core.to_csv(f"{filename_core}.csv", index=True)
print(f"Saved {filename_core}.csv")

# Subset for the n2o-precision-fertilization repository (parquet only).
# Copy it there as data/raw/fluxes_meteo_mgmt.parquet (see README, section 'Downstream use').
filename_n2o = f"90.3_{NAME}_SUBSET_n2o-precision-fertilization"
save_parquet(data=n2o_subset, filename=filename_n2o)

# **End of notebook**

In [ ]:
from datetime import datetime
dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Finished. {dt_string}")